# StateSet Agents — Whitepaper v1.0 GSM8K Benchmark

This notebook produces the **canonical empirical result** for the v1.0 whitepaper revision: fine-tune Qwen 3.5 0.8B on GSM8K with GSPO and measure the pass@1 improvement over the un-fine-tuned baseline.

**Estimated runtime:** ~45 minutes on a Colab A100. **Cost:** ~$0.50.

## What this notebook does

1. Pins the framework to commit `c0dbd68` (the whitepaper v0.12.2 version).
2. Installs the training extras (PyTorch, Transformers, PEFT, TRL, datasets).
3. Sets canonical seed `42` across all RNGs via `set_all_seeds`.
4. Loads GSM8K (200 train / 100 eval).
5. Evaluates the **un-fine-tuned baseline** to establish the floor.
6. Fine-tunes with **GSPO** (the framework's flagship trainer for short-output tasks).
7. Re-evaluates and computes the improvement.
8. Writes a JSON result conforming to `benchmark_results/SCHEMA.md`.
9. (Optional) Uploads the result + W&B run URL.

**Open in Colab:** [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stateset/stateset-agents/blob/master/notebooks/whitepaper_v1_gsm8k_benchmark.ipynb)

**Need an A100?** `Runtime → Change runtime type → A100 GPU`. Free T4 will work but the timings will not match the whitepaper.

## 1. Pin the framework + install

In [ ]:
import os
import subprocess
import sys

PINNED_COMMIT = 'c0dbd68'  # whitepaper v0.12.2 version (was '14c0e65' before v0.12.0 — set_all_seeds didn't exist there)

if not os.path.exists('/content/stateset-agents'):
    subprocess.check_call([
        'git', 'clone', '--quiet',
        'https://github.com/stateset/stateset-agents',
        '/content/stateset-agents'
    ])
subprocess.check_call(['git', '-C', '/content/stateset-agents', 'checkout', '--quiet', PINNED_COMMIT])
%cd /content/stateset-agents
print('Pinned to', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode().strip())

In [ ]:
%pip install --quiet -e '.[training,api]'
%pip install --quiet datasets accelerate bitsandbytes
print('Install complete')

## 2. Lock down reproducibility

One call seeds Python's `random`, NumPy, PyTorch (CPU and CUDA), and Transformers.

In [ ]:
from stateset_agents.utils.reproducibility import set_all_seeds

SEED = 42
state = set_all_seeds(SEED, deterministic_cuda=False)
print('Seeds applied:', state.to_dict())

## 3. Load GSM8K

In [ ]:
from stateset_agents.data.gsm8k import load_gsm8k, make_gsm8k_scenarios, GSM8KReward

N_TRAIN = 200
N_EVAL = 100

train_examples, test_examples = load_gsm8k(limit=max(N_TRAIN, N_EVAL))
train_examples = train_examples[:N_TRAIN]
eval_examples = test_examples[:N_EVAL]

print(f'Train: {len(train_examples)} examples')
print(f'Eval:  {len(eval_examples)} examples')
print('Sample question:')
print(' Q:', train_examples[0].question)
print(' Gold answer:', train_examples[0].gold_answer)

## 4. Baseline evaluation — what does the un-fine-tuned model score?

In [ ]:
import asyncio
from stateset_agents.core.agent import Agent
from stateset_agents.core.agent_config import AgentConfig
from stateset_agents.data.gsm8k import extract_predicted_answer

MODEL_NAME = 'Qwen/Qwen3.5-0.8B'

async def baseline_eval():
    agent = Agent(config=AgentConfig(
        model_name=MODEL_NAME,
        max_new_tokens=256,
        temperature=0.0,
        do_sample=False,
        torch_dtype='bfloat16',
    ))
    await agent.initialize()

    correct = 0
    parseable = 0
    for i, ex in enumerate(eval_examples):
        prompt = f'Solve this step by step.\n\n{ex.question}\n\nAnswer:'
        response = await agent.generate_response(prompt)
        pred = extract_predicted_answer(response)
        if pred is not None:
            parseable += 1
            if abs(pred - ex.gold_answer) < 1e-3:
                correct += 1
        if (i + 1) % 20 == 0:
            print(f'  [{i+1}/{len(eval_examples)}] running pass@1 = {correct/(i+1):.3f}')
    return {
        'pass_at_1': correct / len(eval_examples),
        'parse_rate': parseable / len(eval_examples),
        'n': len(eval_examples),
    }

baseline = await baseline_eval()  # top-level await — Jupyter already has a running loop
print('\nBaseline:', baseline)

## 5. Fine-tune with GSPO

GSPO is the framework's flagship trainer for short-output tasks. Note the **3e-4 / 4e-4 clip range** — this is much tighter than token-level PPO's `0.2` because GSPO's sequence-level importance ratio is already length-normalized. See §5.2 of the whitepaper.

In [ ]:
from stateset_agents.training import GSPOConfig, train_with_gspo
from stateset_agents.core import ConversationEnvironment, MultiTurnAgent

config = GSPOConfig(
    model_name=MODEL_NAME,
    num_generations=4,
    clip_range_left=3e-4,
    clip_range_right=4e-4,
    learning_rate=5e-6,
    max_prompt_length=512,
    max_completion_length=256,
    use_lora=True,
    lora_r=16,
    lora_alpha=32,
    gradient_checkpointing=True,
    num_epochs=1,
    warmup_ratio=0.1,
    output_dir='/content/gspo_gsm8k',
)

agent = MultiTurnAgent(AgentConfig(model_name=MODEL_NAME, torch_dtype='bfloat16'))
# Do NOT call agent.initialize() — train_with_gspo loads the model itself.

env = ConversationEnvironment(
    scenarios=make_gsm8k_scenarios(train_examples),
    reward_fn=GSM8KReward(),
    max_turns=1,
)


import time
t0 = time.time()
trained_agent = await train_with_gspo(
    config=config,
    agent=agent,
    environment=env,
    reward_model=env.reward_fn,
)
train_wall_clock = time.time() - t0
print(f'\nTraining wall-clock: {train_wall_clock:.1f}s ({train_wall_clock/60:.1f}m)')

## 6. Post-training evaluation

In [ ]:
async def final_eval():
    correct = 0
    parseable = 0
    for i, ex in enumerate(eval_examples):
        prompt = f'Solve this step by step.\n\n{ex.question}\n\nAnswer:'
        response = await agent.generate_response(prompt)
        pred = extract_predicted_answer(response)
        if pred is not None:
            parseable += 1
            if abs(pred - ex.gold_answer) < 1e-3:
                correct += 1
        if (i + 1) % 20 == 0:
            print(f'  [{i+1}/{len(eval_examples)}] running pass@1 = {correct/(i+1):.3f}')
    return {
        'pass_at_1': correct / len(eval_examples),
        'parse_rate': parseable / len(eval_examples),
        'n': len(eval_examples),
    }

final = await final_eval()  # top-level await — Jupyter already has a running loop
print('\nFinal:', final)
print(f'Improvement: {final["pass_at_1"] - baseline["pass_at_1"]:+.3f}')

## 7. Save result JSON (whitepaper-compatible)

In [ ]:
import json
import subprocess
import torch
from datetime import datetime, timezone
from pathlib import Path

result = {
    'trainer': 'gspo',
    'model': MODEL_NAME,
    'seed': SEED,
    'commit': PINNED_COMMIT,
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'config': {
        'num_generations': config.num_generations,
        'clip_range_left': config.clip_range_left,
        'clip_range_right': config.clip_range_right,
        'learning_rate': config.learning_rate,
        'lora_r': config.lora_r,
    },
    'metrics': {
        'eval_pass_at_1': final['pass_at_1'],
        'eval_pass_at_1_baseline': baseline['pass_at_1'],
        'improvement': final['pass_at_1'] - baseline['pass_at_1'],
        'eval_parse_rate': final['parse_rate'],
        'eval_parse_rate_baseline': baseline['parse_rate'],
        'wall_clock_seconds': train_wall_clock,
        'train_examples': N_TRAIN,
        'eval_examples': N_EVAL,
        'peak_vram_mb': torch.cuda.max_memory_allocated() // (1024**2) if torch.cuda.is_available() else 0,
    },
    'hardware': {
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'cuda': torch.version.cuda if torch.cuda.is_available() else None,
    }
}

out = Path(f'/content/gspo_seed{SEED}_qwen3_5_0_8b.json')
out.write_text(json.dumps(result, indent=2))
print('Result:')
print(json.dumps(result, indent=2))
print(f'\nSaved to: {out}')

## 8. Next steps

- **Run with two more seeds** (1337 and 2026) to get variance bars — the v1.0 whitepaper requires 3 seeds per config.
- **Repeat for GRPO and DAPO** by changing the trainer import — same notebook, three configs.
- **Upload the JSON** to the whitepaper repo under `benchmark_results/whitepaper_v1/`.
- **Submit a PR** updating §11.7 of the whitepaper with your numbers.

If your `improvement` is at least 0.03 with all three seeds showing the same sign, the result passes the v1.0 publication gates (see `benchmark_results/SCHEMA.md`).